# 03 – Baseline

Before building a model, I need a simple benchmark to beat. The baseline prices each car at the
**median price of the same make/model/year** in the training data, the "comparable sales" approach
a person would use when browsing listings. It ignores mileage, accidents and every other feature.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_parquet("../data/cars_clean.parquet")

### Train/test split
80% of listings are used to calculate the medians, and 20% are held back as unseen "new" listings
for grading. random_state=42 is used in every notebook so all models are graded on the same test cars.

In [2]:
train, test = train_test_split(df, test_size=0.2, random_state = 42)
print(train.shape, test.shape)

(599053, 20) (149764, 20)


### Baseline prediction
Use the median price of the same make/model/year. If that combination isn't in the training data,
fall back to make/model, then make, then the overall median, so every test car gets a prediction.

In [3]:
def group_median(train, test, keys):
    med = train.groupby(keys, observed=True)["price"].median().rename("pred")
    return test.join(med, on=keys)["pred"]

pred = group_median(train, test, ["manufacturer", "model", "year"])
pred = pred.fillna(group_median(train, test, ["manufacturer", "model"]))
pred = pred.fillna(group_median(train, test, ["manufacturer"]))
pred = pred.fillna(train["price"].median())

In [5]:
err = (pred - test["price"]).abs()
print(f"MAE: ${err.mean():,.0f}")
print(f"MAPE: {(err / test["price"]).mean():.1%}")
print(f"Median abs error: {err.median():,.0f}")

MAE: $3,070
MAPE: 12.1%
Median abs error: 2,004


### Result
| Metric | Baseline |
|---|---|
| MAE | $3,070 |
| MAPE | 12.1% |
| Median absolute error | $2,004 |

This is the score to beat. The median is lower than the MAE, so most cars are priced reasonably
well, but a smaller group has large misses, most likely cars with unusually high or low mileage,
which the baseline ignores completely.